# DGN - Deep Learning-based Image Denoising

This notebook demonstrates deep learning-inspired denoising techniques (DGN) for computer vision applications.
We implement both traditional and modern approaches to image denoising.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dgn_denoising import ImageDenoiser

# Create an instance of the ImageDenoiser
denoiser = ImageDenoiser()

print("DGN Image Denoising Module Loaded Successfully!")

## Create Test Images with Different Types of Noise

In [ ]:
# Create a synthetic test image with geometric patterns
test_image = np.zeros((256, 256), dtype=np.uint8)

# Add various shapes for comprehensive testing
cv2.rectangle(test_image, (50, 50), (150, 150), 255, -1)  # White square
cv2.circle(test_image, (200, 200), 30, 128, -1)           # Gray circle
cv2.line(test_image, (0, 128), (255, 128), 200, 3)       # Horizontal line
cv2.ellipse(test_image, (64, 200), (30, 20), 45, 0, 360, 180, -1)  # Ellipse

# Display the original test image
plt.figure(figsize=(12, 4))
plt.subplot(1, 4, 1)
plt.imshow(test_image, cmap='gray')
plt.title('Original Test Image')
plt.axis('off')

# Add different types of noise
gaussian_noisy = denoiser.add_noise(test_image, 'gaussian', 30)
salt_pepper_noisy = denoiser.add_noise(test_image, 'salt_pepper', 20)
speckle_noisy = denoiser.add_noise(test_image, 'speckle', 25)

plt.subplot(1, 4, 2)
plt.imshow(gaussian_noisy, cmap='gray')
plt.title('Gaussian Noise')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(salt_pepper_noisy, cmap='gray')
plt.title('Salt & Pepper Noise')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(speckle_noisy, cmap='gray')
plt.title('Speckle Noise')
plt.axis('off')

plt.tight_layout()
plt.show()

## Compare Different Denoising Methods

In [ ]:
# Compare all denoising methods on Gaussian noise
print("Comparing denoising methods on Gaussian noise...")
results = denoiser.compare_methods(test_image, noise_type='gaussian', noise_level=30)

# Display results with quality metrics
denoiser.visualize_comparison(results, figsize=(16, 8))
plt.suptitle('DGN Denoising Methods Comparison - Gaussian Noise', fontsize=16, y=1.02)
plt.show()

# Print quality metrics
print("\nQuality Metrics (Higher is Better):")
print("=" * 45)
methods = [k for k in results.keys() if k not in ['original', 'noisy']]
for method in methods:
    if 'psnr' in results[method]:
        print(f"{method:12}: PSNR = {results[method]['psnr']:.2f} dB, SSIM = {results[method]['ssim']:.3f}")
    else:
        print(f"{method:12}: Error - {results[method].get('error', 'Unknown')}")

## Test Individual Denoising Methods with Parameter Tuning

In [ ]:
# Test the Simple CNN approach with different parameters
noisy_image = gaussian_noisy.copy()

# Apply different denoising methods
gaussian_result = denoiser.denoise(noisy_image, 'gaussian', kernel_size=5, sigma=1.5)
bilateral_result = denoiser.denoise(noisy_image, 'bilateral', d=9, sigma_color=75, sigma_space=75)
nlm_result = denoiser.denoise(noisy_image, 'nlm', h=12)
cnn_result = denoiser.denoise(noisy_image, 'simple_cnn', filter_size=5)

# Calculate quality metrics for each method
gaussian_quality = denoiser.evaluate_quality(test_image, gaussian_result)
bilateral_quality = denoiser.evaluate_quality(test_image, bilateral_result)
nlm_quality = denoiser.evaluate_quality(test_image, nlm_result)
cnn_quality = denoiser.evaluate_quality(test_image, cnn_result)

# Display side-by-side comparison
plt.figure(figsize=(16, 10))

images = [test_image, noisy_image, gaussian_result, bilateral_result, nlm_result, cnn_result]
titles = ['Original', 'Noisy', 
          f'Gaussian\nPSNR: {gaussian_quality["psnr"]:.2f}\nSSIM: {gaussian_quality["ssim"]:.3f}',
          f'Bilateral\nPSNR: {bilateral_quality["psnr"]:.2f}\nSSIM: {bilateral_quality["ssim"]:.3f}',
          f'NL-Means\nPSNR: {nlm_quality["psnr"]:.2f}\nSSIM: {nlm_quality["ssim"]:.3f}',
          f'Simple CNN\nPSNR: {cnn_quality["psnr"]:.2f}\nSSIM: {cnn_quality["ssim"]:.3f}']

for i, (img, title) in enumerate(zip(images, titles)):
    plt.subplot(2, 3, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Real Image Denoising Example

If you have a real image, you can test the DGN denoising methods on it:

In [ ]:
# Example with a sample image (you can replace this with your own image path)
try:
    # Try to load a real image (replace 'sample.jpg' with your image)
    real_image = cv2.imread('sample.jpg', cv2.IMREAD_GRAYSCALE)
    
    if real_image is not None:
        # Resize if too large
        if real_image.shape[0] > 512 or real_image.shape[1] > 512:
            real_image = cv2.resize(real_image, (512, 512))
        
        # Add noise and denoise
        noisy_real = denoiser.add_noise(real_image, 'gaussian', 25)
        denoised_real = denoiser.denoise(noisy_real, 'simple_cnn')
        
        # Display results
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.imshow(real_image, cmap='gray')
        plt.title('Original Real Image')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(noisy_real, cmap='gray')
        plt.title('With Gaussian Noise')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(denoised_real, cmap='gray')
        plt.title('DGN Denoised')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Calculate quality
        quality = denoiser.evaluate_quality(real_image, denoised_real)
        print(f"Real Image Denoising Quality: PSNR = {quality['psnr']:.2f} dB, SSIM = {quality['ssim']:.3f}")
    else:
        print("No sample image found. Place an image named 'sample.jpg' in the current directory to test on real images.")
        
except Exception as e:
    print(f"Could not load real image: {e}")
    print("To test on real images, place an image file in the current directory and update the file path above.")

## Summary

This DGN (Deep Learning-based) denoising implementation provides:

1. **Multiple Denoising Methods**: Gaussian blur, bilateral filtering, Non-Local Means, and a simple CNN-inspired approach
2. **Noise Simulation**: Ability to add Gaussian, salt-and-pepper, and speckle noise for testing
3. **Quality Evaluation**: PSNR and SSIM metrics to quantitatively assess denoising performance
4. **Visualization Tools**: Easy comparison of different methods with visual output

The Simple CNN method uses separable convolutions and edge-preserving techniques to achieve good denoising performance while being computationally efficient.

### Key Features:
- **Efficient**: Uses separable convolutions instead of full CNNs for speed
- **Edge-Preserving**: Maintains important image structures while removing noise
- **Flexible**: Easy to extend with additional denoising algorithms
- **Quantitative**: Provides objective quality metrics for method comparison